# 实验七 决策树

决策树通过一系列"是否满足条件"的判断完成分类，是一种有很好可解释性的非参数学习算法。

本实验先使用 Iris 数据完成基础决策树分类与树结构观察，再使用带噪声的月牙形数据调整剪枝参数。每次只修改一个参数，记录训练准确率、测试准确率、树深度、叶节点数量和决策边界变化。

## 1. 导入环境并准备 Iris 数据

使用 Iris 花瓣长度和花瓣宽度两个特征建立决策树。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from matplotlib.colors import ListedColormap

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

iris = datasets.load_iris()
X = iris.data[:, 2:4]       # 花瓣长度、花瓣宽度
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=35)
plt.xlabel('花瓣长度')
plt.ylabel('花瓣宽度')
plt.title('Iris 数据分布')
plt.show()

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'sklearn'

> **操作要求**：记录训练集和测试集样本数量，并观察三类样本在二维空间中的分布。

## 2. 绘制决策边界并训练基础模型

训练 `max_depth=2` 的基础决策树，绘制决策边界和树结构。

In [ ]:
def plot_decision_boundary(model, X, y, title, axis):
    x0, x1 = np.meshgrid(
        np.linspace(axis[0], axis[1], int((axis[1] - axis[0]) * 100)).reshape(-1, 1),
        np.linspace(axis[2], axis[3], int((axis[3] - axis[2]) * 100)).reshape(-1, 1),
    )
    X_new = np.c_[x0.ravel(), x1.ravel()]
    zz = model.predict(X_new).reshape(x0.shape)
    cmap = ListedColormap(['#EF9A9A', '#FFF59D', '#90CAF9'])
    plt.figure(figsize=(8, 6))
    plt.contourf(x0, x1, zz, cmap=cmap, alpha=0.6)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=35, edgecolors='k')
    plt.xlabel('花瓣长度')
    plt.ylabel('花瓣宽度')
    plt.title(title)
    plt.show()


model = DecisionTreeClassifier(max_depth=2, criterion='gini', random_state=42)
model.fit(X_train, y_train)
print('训练准确率：', model.score(X_train, y_train))
print('测试准确率：', model.score(X_test, y_test))
print('树深度：', model.get_depth())
print('叶节点数：', model.get_n_leaves())

plot_decision_boundary(model, X, y, '基础决策树：max_depth=2', axis=[0.5, 7.5, 0.0, 3.0])

plt.figure(figsize=(10, 6))
plot_tree(model, filled=True, feature_names=iris.feature_names, class_names=iris.target_names)
plt.show()

> **操作要求**：观察根节点使用的特征与阈值，并记录树深度、叶节点数量和测试准确率。

## 3. 比较 gini 与 entropy

分别设置 `criterion="gini"` 和 `criterion="entropy"`，比较准确率和树结构。

In [ ]:
for criterion_value in ['gini', 'entropy']:
    model = DecisionTreeClassifier(max_depth=3, criterion=criterion_value, random_state=42)
    model.fit(X_train, y_train)
    print('\ncriterion =', criterion_value)
    print('训练准确率：', model.score(X_train, y_train))
    print('测试准确率：', model.score(X_test, y_test))
    print('树深度：', model.get_depth())
    print('叶节点数：', model.get_n_leaves())

> **操作要求**：比较两种划分准则的准确率和树结构，说明结果是否存在明显差异。

## 4. 生成月牙形数据并调整 max_depth

使用带噪声的月牙形数据（`make_moons`），将 `max_depth` 设置为 1、2、4 和 None，观察欠拟合与过拟合现象。

In [ ]:
from sklearn.datasets import make_moons

X_moon, y_moon = make_moons(n_samples=300, noise=0.25, random_state=666)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_moon, y_moon, test_size=0.25, random_state=42, stratify=y_moon
)

for depth in [1, 2, 4, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(Xm_train, ym_train)
    print('\nmax_depth =', depth)
    print('训练准确率：', model.score(Xm_train, ym_train))
    print('测试准确率：', model.score(Xm_test, ym_test))
    print('实际深度：', model.get_depth())
    print('叶节点数：', model.get_n_leaves())
    plot_decision_boundary(model, X_moon, y_moon, f'max_depth={depth}', axis=[-1.5, 2.5, -1.0, 1.5])

> **操作要求**：判断 max_depth 过小时是否欠拟合，未限制深度时边界是否过于曲折。

## 5. 修改 min_samples_split 和 min_samples_leaf

分别修改 `min_samples_split` 和 `min_samples_leaf`，观察边界是否更加平滑。

In [ ]:
for split_value in [2, 10, 30]:
    model = DecisionTreeClassifier(min_samples_split=split_value, random_state=42)
    model.fit(Xm_train, ym_train)
    print('min_samples_split =', split_value,
          '测试准确率 =', model.score(Xm_test, ym_test),
          '深度 =', model.get_depth(),
          '叶节点数 =', model.get_n_leaves())

for leaf_value in [1, 5, 15]:
    model = DecisionTreeClassifier(min_samples_leaf=leaf_value, random_state=42)
    model.fit(Xm_train, ym_train)
    print('min_samples_leaf =', leaf_value,
          '测试准确率 =', model.score(Xm_test, ym_test),
          '深度 =', model.get_depth(),
          '叶节点数 =', model.get_n_leaves())

> **操作要求**：每次只调整一个参数，观察参数增大后树深度和叶节点数量是否减少。

## 6. 修改 max_leaf_nodes

将 `max_leaf_nodes` 设置为 2、4、8，比较模型复杂度和分类效果。

In [ ]:
for leaf_nodes in [2, 4, 8]:
    model = DecisionTreeClassifier(max_leaf_nodes=leaf_nodes, random_state=42)
    model.fit(Xm_train, ym_train)
    print('\nmax_leaf_nodes =', leaf_nodes)
    print('训练准确率：', model.score(Xm_train, ym_train))
    print('测试准确率：', model.score(Xm_test, ym_test))
    print('树深度：', model.get_depth())
    print('叶节点数：', model.get_n_leaves())
    plot_decision_boundary(model, X_moon, y_moon, f'max_leaf_nodes={leaf_nodes}', axis=[-1.5, 2.5, -1.0, 1.5])

> **操作要求**：比较三组决策边界，选择一组兼顾测试准确率和边界简洁性的参数。

## 7. 创新拓展实践：手写数字十分类

使用手写数字数据集完成 0～9 十分类实验。

任务：比较 `max_depth=5、10、None` 的训练准确率和测试准确率，再将 `min_samples_leaf` 修改为 1、2、4，选择一组较合适的参数，并分析是否出现过拟合。

In [ ]:
digits = datasets.load_digits()
X_digit = digits.data
y_digit = digits.target
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_digit, y_digit, test_size=0.2, random_state=42, stratify=y_digit
)

for depth in [5, 10, None]:
    model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=2, random_state=42)
    model.fit(Xd_train, yd_train)
    print(depth, model.score(Xd_train, yd_train), model.score(Xd_test, yd_test))